# GPU-Accelerated Spectral Classification (cuML + Optuna)

## Introduction

In this extended version of the SDSS spectral classification pipeline, we explore how GPU acceleration can significantly speed up model training and hyperparameter optimization.

We reuse the same astronomical dataset (SDSS DR16 spectra), but shift computation to **RAPIDS cuDF/cuML** and **GPU-enabled XGBoost**, enabling faster experimentation on large-scale spectral data.

The overall workflow remains unchanged:

* preprocessing (L2 normalization)
* dimensionality reduction (optional PCA)
* supervised classification (Logistic Regression / Random Forest / XGBoost)
* hyperparameter optimization (Optuna)
* final evaluation on held-out test data

The main difference is computational efficiency: most heavy operations are executed on the GPU.


## Imports (GPU Stack)


In [ ]:
import time
import numpy as np
import pandas as pd
import joblib
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import cudf
import cupy as cp
import cuml
import xgboost as xgb

from cuml.pipeline import Pipeline
from cuml.preprocessing import Normalizer
from cuml.decomposition import PCA
from cuml.linear_model import LogisticRegression
from cuml.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load Dataset

The Sloan Digital Sky Survey (SDSS) is a major astronomical survey that has been operating at Apache Point Observatory since 2000.
Up-to-date information can be found here: https://www.sdss.org/.

In this tutorial we use optical spectra obtained by the SDSS spectroscopic survey. The data contained in the `data_spectra/` folder is part of SDSS Data Release 16 (DR16).


In [ ]:
data_path = "../data/spectra/"

data = np.load(data_path + "data.npy")
labels = np.load(data_path + "labels.npy")
wavelengths = np.load(data_path + "wavelengths.npy")

class_names = ["AGN", "galaxy", "QSO"]

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)
print("Wavelength bins:", wavelengths.shape)

To gain an initial qualitative understanding of the dataset, we randomly visualize one spectrum from each astrophysical class.

This allows us to inspect the overall continuum shape and prominent spectral features characteristic of galaxies, AGN, and quasars.

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(12,3))
rng = np.random.default_rng(12345)
for i,cls in enumerate(np.unique(labels)):
    idx = np.where(labels == cls)[0]
    n = rng.choice(idx)
    axs[i].plot(wavelengths, data[n], label=f"{class_names[labels[n]]}")
    axs[i].set_xlabel('wavelength(Å)')
    axs[i].set_title(f"{n} - {class_names[labels[n]]}")
    
axs[0].set_ylabel('flux (10-17 erg/s/cm2/Å)')
plt.tight_layout()
plt.show()


To better characterize the dataset, we compute the mean spectrum and standard deviation for each astrophysical class. 

This provides a compact representation of the typical spectral behavior and variability across wavelengths for galaxies, AGN, and quasars.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12))
axs = axs.ravel()
colors = ["tab:blue", "tab:red", "tab:green"]
for i in (0, 1, 2):
    idx = np.where(labels == i)[0]
    mu = data[idx].mean(0)
    std = data[idx].std(0)
    l = axs[i].plot(wavelengths, mu, label=f"{class_names[i]}", color=colors[i])
    axs[i].fill_between(wavelengths, mu - std, mu + std, color=colors[i], alpha=0.3)
    axs[i].set_xlabel('wavelength (Å)')
    axs[i].set_ylim(-40, 80)
    axs[i].set_title(f"{class_names[i]}")
    
axs[0].set_ylabel('flux (10-17 erg/s/cm2/Å)')
axs[2].set_ylabel('flux (10-17 erg/s/cm2/Å)')
axs[-1].plot(wavelengths, data.mean(0), label=f"mean", color="black")
axs[-1].fill_between(wavelengths, data.mean(0) - data.std(0), data.mean(0) + data.std(0), color="grey", alpha=0.3)
axs[-1].set_xlabel('wavelength (Å)')
axs[-1].set_ylim(-40, 80)
axs[-1].set_title("mean")
plt.tight_layout()
plt.show()


We visualize the class distribution of the dataset to inspect the relative number of samples belonging to each astrophysical category.

In [ ]:
# Maps the integer labels to their actual string names for the plot
mapped_labels = [class_names[i] for i in labels]
sns.countplot(x=mapped_labels, order=class_names)
plt.title("Class Distribution")
plt.show()

## Data-driven Classification Pipeline

The dataset is divided into a training set and an independent test set using stratified sampling in order to preserve the original class distribution. 

Model selection and hyperparameter optimization are performed exclusively on the training data through cross-validation, while the test set remains completely unseen until the final evaluation stage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels.astype("int32"),
    test_size=0.1,
    stratify=labels,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Move Data to GPU (cuDF / cuPy)

We convert NumPy arrays into GPU-friendly formats.

In [ ]:
X_train_gdf = cudf.DataFrame(
    X_train.astype(np.float32)
)

y_train_gdf = cudf.Series(
    y_train.astype(np.int32)
)

X_test_gdf = cudf.DataFrame(
    X_test.astype(np.float32)
)

y_test_gdf = cudf.Series(
    y_test.astype(np.int32)
)

Because the spectra correspond to objects spanning a wide range of intrinsic luminosities and cosmological distances, their observed flux amplitudes can vary substantially. 

To reduce the impact of these global intensity differences and emphasize spectral shape, each spectrum is first normalized using L2 normalization before applying Principal Component Analysis (PCA).



In [ ]:
pipeline_pca = Pipeline([
    ("normalize", Normalizer(norm="l2")),
    ("pca", PCA(n_components=5, random_state=42))
])

X_proj = pipeline_pca.fit_transform(X_train)


We project the normalized spectra onto the first two principal components obtained from PCA and visualize the resulting 2D embedding. 

Each point represents one spectrum, colored by its true class, allowing us to assess how well different astrophysical populations separate in the reduced feature space.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

scatter = ax.scatter(
    X_proj[:, 0],
    X_proj[:, 1],
    c=y_train,
    s=4,
    cmap="viridis",
    alpha=0.8
)

cbar = fig.colorbar(scatter, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(class_names)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA Projection of SDSS Spectra")
plt.tight_layout()
plt.show()

Since PCA is applied to L2-normalized spectra, the resulting components describe variations in *spectral shape* rather than absolute flux levels.

The first principal component primarily captures large-scale continuum variations across wavelength, which are closely related to the overall spectral slope or “color” of the object.

The second component encodes more localized structure, including variations in emission and absorption features as well as contributions from the 4000 Å break.

Together, these eigenspectra provide a compact, interpretable basis for understanding dominant modes of spectral variation. However, PCA is a linear method and is not optimized for class separation, particularly when differences are driven by narrow spectral features.

> C.W. Yip et al., *Spectral Classification of Quasars in the Sloan Digital Sky Survey: Eigenspectra, Redshift, and Luminosity Effects*, Astronomical Journal 128:6 (2004).

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4))
axs[1].plot(wavelengths, pipeline_pca.named_steps["pca"].mean_, label=f"Mean")
for i in range(2):
    l = axs[1].plot(wavelengths, pipeline_pca.named_steps["pca"].components_[i], label=f"Component {(i + 1)}")
    c = l[0].get_color()

axs[1].set_xlabel('Wavelength (Å)')
axs[1].set_ylabel('Scaled flux')
axs[1].set_title('Mean Spectrum and Eigen-spectra')
axs[1].legend()
axs[0].bar(np.arange(1, 6), pipeline_pca.named_steps["pca"].explained_variance_ratio_)

axs[0].set_xlabel("Principal Component")
axs[0].set_ylabel("Explained Variance Ratio")
plt.show()


## Classification Pipeline and Model Selection

We construct a flexible scikit-learn pipeline that combines optional preprocessing, dimensionality reduction via PCA, and a downstream classifier. We then define a unified hyperparameter search space to compare Logistic Regression, Random Forest, and XGBoost models under identical preprocessing conditions using randomized cross-validation search.

In [ ]:
## Objective Function

Key idea:

* cuML handles preprocessing + PCA + some models
* XGBoost runs on CUDA (`tree_method="hist", device="cuda"`)


In [ ]:
def objective(trial, X, y, cv, model_type):

    # PCA configuration
    use_pca = trial.suggest_categorical("use_pca", [True, False])

    if use_pca:
        # cuML PCA does not reliably support sklearn-style
        # explained variance ratios such as 0.95
        # therefore we optimize integer component counts
        pca_n = trial.suggest_int("pca_n_components", 10, 100, 1000)
        pca = PCA(n_components=pca_n,)
    else:
        pca = "passthrough"

    # Model selection

    if model_type == "logit":
        clf = LogisticRegression(
            C=trial.suggest_float(
                "logit_C",
                1e-1,
                1e4,
                log=True
            ),
            max_iter=10000,
            solver="qn"
        )

    elif model_type == "rf":

        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int(
                "rf_n_estimators",
                100,
                400
            ),

            max_depth=trial.suggest_categorical(
                "rf_max_depth",
                [5, 10, 20, None]
            ),

            max_features=trial.suggest_categorical(
                "rf_max_features",
                ["sqrt", "log2"]
            ),

            min_samples_leaf=trial.suggest_categorical(
                "rf_min_samples_leaf",
                [1, 2, 5]
            ),

        )

    elif model_type == "xgb":

        clf = xgb.XGBClassifier(

            objective="multi:softprob",
            eval_metric="mlogloss",

            tree_method="hist",
            device="cuda",

            n_estimators=trial.suggest_int(
                "xgb_n_estimators",
                100,
                500
            ),

            learning_rate=trial.suggest_float(
                "xgb_learning_rate",
                1e-3,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_categorical(
                "xgb_max_depth",
                [3, 4, 5, 6]
            ),

            subsample=trial.suggest_categorical(
                "xgb_subsample",
                [0.6, 0.8, 1.0]
            ),

            colsample_bytree=trial.suggest_categorical(
                "xgb_colsample_bytree",
                [0.5, 0.8, 1.0]
            ),

            min_child_weight=trial.suggest_categorical(
                "xgb_min_child_weight",
                [1, 3, 5]
            )
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    # Pipeline

    steps = [("normalize", Normalizer(norm="l2"))]

    if pca != "passthrough":
        steps.append(("pca", pca))

    steps.append(("classifier", clf))

    pipe = Pipeline(steps)

    # Cross-validation

    scores = []

    # sklearn splitter operates on CPU arrays
    for train_idx, val_idx in cv.split(X.to_pandas(), y.to_pandas()):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        # fit
        pipe.fit(X_train_fold, y_train_fold)
        # predict
        preds = pipe.predict(X_val_fold)
        # move to CPU for sklearn metrics
        # Convert ground truth safely
        y_true = y_val_fold.to_numpy()
        # Convert predictions safely
        if hasattr(preds, "to_numpy"):
            y_pred = preds.to_numpy()
        elif isinstance(preds, cp.ndarray):
            y_pred = cp.asnumpy(preds)
        else:
            y_pred = np.asarray(preds)
        
        score = f1_score(y_true, y_pred, average="macro")
        scores.append(score)

    return float(np.mean(scores))

## Cross-Validation Strategy

We use stratified k-fold cross-validation to evaluate models in a robust and balanced way, ensuring that each fold preserves the original class distribution. 

This helps provide a more reliable estimate of model performance across the different astrophysical classes.


In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
)

# Hyperparameter Optimization

We perform model selection by iterating over different classifier families within a shared pipeline and optimizing their hyperparameters using randomized cross-validation. 

For each model, we train using stratified k-fold CV, evaluate performance with macro-averaged F1 score, and finally assess generalization on the held-out test set. 

The best-performing configuration is tracked based on cross-validation performance.

In [ ]:
all_results = {}

best_overall_score = -np.inf
best_overall_model = None
best_model_name = None

for model_name in ["logit", "rf", "xgb"]:

    print("\n" + "=" * 60)
    print(f"Optimizing {model_name}")
    print("=" * 60)

    study = optuna.create_study(
        direction="maximize"
    )

    start = time.time()

    study.optimize(
        lambda trial: objective(
            trial,
            X_train_gdf,
            y_train_gdf,
            cv,
            model_name
        ),
        n_trials=20
    )

    elapsed = time.time() - start

    print("\nBest CV F1:")
    print(f"{study.best_value:.4f}")

    print("\nBest Parameters:")
    print(study.best_params)

    all_results[model_name] = {
        "best_score": study.best_value,
        "best_params": study.best_params,
        "fit_time_sec": elapsed
    }

    # track best overall model
    if study.best_value > best_overall_score:

        best_overall_score = study.best_value
        best_model_name = model_name

## Train Final Model

In [ ]:
best_params = all_results[best_model_name]["best_params"]

print("\nBest overall model:")
print(best_model_name)

steps = [
    ("normalize", Normalizer(norm="l2"))
]

# PCA
if best_params.get("use_pca", False):

    steps.append((
        "pca",
        PCA(
            n_components=best_params["pca_n_components"],
        )
    ))

# Classifier
if best_model_name == "logit":

    clf = LogisticRegression(
        C=best_params["logit_C"],
        max_iter=10000,
        solver="qn"
    )

elif best_model_name == "rf":

    clf = RandomForestClassifier(
        n_estimators=best_params["rf_n_estimators"],
        max_depth=best_params["rf_max_depth"],
        max_features=best_params["rf_max_features"],
        min_samples_leaf=best_params["rf_min_samples_leaf"],
    )

elif best_model_name == "xgb":

    clf = xgb.XGBClassifier(

        objective="multi:softprob",
        eval_metric="mlogloss",

        tree_method="hist",
        device="cuda",


        n_estimators=best_params["xgb_n_estimators"],
        learning_rate=best_params["xgb_learning_rate"],
        max_depth=best_params["xgb_max_depth"],
        subsample=best_params["xgb_subsample"],
        colsample_bytree=best_params["xgb_colsample_bytree"],
        min_child_weight=best_params["xgb_min_child_weight"]
    )

steps.append(("classifier", clf))

best_model = Pipeline(steps)

In [ ]:
best_model.fit(
    X_train_gdf,
    y_train_gdf
)

## Save Model

We save the best-performing trained pipeline to disk using `joblib` for later reuse. 

This includes the full preprocessing, dimensionality reduction, and classification steps, allowing the model to be directly reloaded for inference without retraining.

In [ ]:
joblib.dump(
    best_model,
    "best_spectrum_classifier_gpu_optuna.pkl"
)

## Evaluation on Test Set

In [ ]:
# We load the best refitted model
final_model = joblib.load("best_spectrum_classifier_gpu_optuna.pkl")


In [ ]:
test_predictions = final_model.predict(
    X_test_gdf
)

# move back to CPU
y_test_cpu = y_test_gdf.to_numpy()
pred_cpu = test_predictions

test_accuracy = accuracy_score(
    y_test_cpu,
    pred_cpu
)

test_f1 = f1_score(
    y_test_cpu,
    pred_cpu,
    average="macro"
)

print(f"Final Test Accuracy: {test_accuracy:.4f}")
print(f"Final Test F1 Macro: {test_f1:.4f}")

In [ ]:
print(
    classification_report(
        y_test_cpu,
        pred_cpu,
        target_names=class_names
    )
)

## Define the Evaluation Metrics

We define evaluation utilities to assess classifier performance in detail. 

In addition to standard classification metrics, we compute per-class sensitivity and specificity from the confusion matrix, and visualize normalized confusion matrices to better understand class-wise prediction behavior.

In [ ]:
def get_multiclass_metrics(y_actual, y_pred, classes):
    cm = confusion_matrix(y_actual, y_pred)
    
    for i, cls in enumerate(classes):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - TP - FP - FN

        # Safe division to prevent ZeroDivisionError
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        print(f"{cls:10s} Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f}")

def report(y_actual, y_pred, classes, title):
    print(classification_report(y_actual, y_pred, target_names=classes))
    print("\nPer-class metrics:")
    get_multiclass_metrics(y_actual, y_pred, classes)
    cm = confusion_matrix(y_actual, y_pred, normalize="true")

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="viridis", 
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

## Confusion Matrix Plot (GPU Model)

In [ ]:
report(y_test, test_predictions, class_names,
       title="Final Test Confusion Matrix")


# Summary

This GPU version reproduces the full ML pipeline while significantly accelerating:

* PCA (cuML)
* preprocessing (cuDF / cuPy)
* classification (cuML + XGBoost CUDA)
* hyperparameter search (Optuna)

The structure remains identical to the CPU pipeline, ensuring direct comparability of results.
